# Exploratory Data Analysis

This notebook is the reproducible EDA entry point for the C-OPN Parkinson classification case study.

Goals:
- inventory each CSV in `data/ssc_data/`
- identify the cohort label columns for `PD`, `AP`, and `HC`
- quantify sparsity and data quality issues
- identify clinically grounded candidate feature groups for downstream modeling

Notes:
- the current workspace snapshot is baseline-only by `Project key`
- the formal data dictionary mentioned in the brief is not available in this workspace, so implementation-level mapping below is provisional

In [ ]:
from pathlib import Path

import pandas as pd

from data.data_preprocessing import (
    DATA_DIR,
    audit_normalized_column_collisions,
    describe_csv_collection,
    load_feature_tables,
    load_target_metadata,
    prepare_modeling_dataset,
    repository_table_catalog,
    select_level_1_or_2_tables,
    summarize_feature_missingness,
)
from scripts.feature_selection import run_feature_selection_experiment

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

DATA_DIR

In [ ]:
catalog = repository_table_catalog()
header_summary = describe_csv_collection(headers_only=True)

catalog.merge(
    header_summary[["filename", "column_count", "preview_columns"]],
    on="filename",
    how="left",
).sort_values("filename").reset_index(drop=True)

## Column Normalization Audit

Several C-OPN files contain headers that differ only by punctuation or trailing spaces. That matters because a naive normalization pass can collapse them into the same feature name.

The preprocessing code now keeps the CSV filename as a prefix and appends a collision suffix like `__dup2` when two columns from the same source file would otherwise normalize to the same feature id.

In [ ]:
collision_audit = audit_normalized_column_collisions()
raw_feature_table = load_feature_tables()

print("Potential raw header normalization collisions")
display(collision_audit)

print("Duplicate merged feature names after the fix:", int(raw_feature_table.columns.duplicated().sum()))

## Target Variables And Candidate Feature Sets

Primary target source:
- `enrollement.csv` -> `Enrolment Group`, mapped to `PD`, `AP`, and `HC`

Secondary subtype source:
- `clinical.csv` -> atypical diagnosis detail fields such as `PSP`, `MSA`, `CBS`, `DLB`, `ET`, and `RBD`

Working recommendation:
- use `PD` vs `AP` as the default binary target
- keep `HC` for optional multiclass experiments
- treat diagnosis fields in `clinical.csv` as metadata only, not model features, to avoid target leakage

Implementation-level guidance:
- likely `Level 1-2`: demographics, epidemiology, questionnaire scales, quality-of-life, fatigue, anxiety, apathy, autonomic symptoms
- likely `Level 3`: MoCA, UPDRS, Timed Up and Go, medication review, clinical history
- likely `Level 4`: detailed neuropsychological batteries

Because the formal data dictionary is not present here, these level assignments should be treated as provisional rather than authoritative.

In [ ]:
targets = load_target_metadata()

print("Multiclass cohort counts")
display(targets["target_multiclass"].value_counts(dropna=False).rename_axis("class").to_frame("n"))

print("\nBinary cohort counts")
display(targets["target_binary"].value_counts(dropna=False).rename_axis("class").to_frame("n"))

print("\nAP subtype detail available in clinical.csv")
display(targets["clinical_subtype"].value_counts(dropna=False).rename_axis("subtype").to_frame("n"))

print("\nSite counts")
display(targets["site"].value_counts(dropna=False).rename_axis("site").to_frame("n").head(15))

In [ ]:
full_summary = describe_csv_collection(headers_only=False)
full_summary.sort_values(["cell_missing_pct", "column_missing_ge_50_pct"], ascending=[False, False]).reset_index(drop=True)

In [ ]:
prepared_binary = prepare_modeling_dataset(target="binary")

print("Binary modeling matrix shape:", prepared_binary.X.shape)
print("Binary class balance:")
display(prepared_binary.y.value_counts().rename_axis("class").to_frame("n"))

print("Top missing features before additional feature engineering:")
display(summarize_feature_missingness(prepared_binary.master, top_n=20))

print("Most implementation-friendly available tables (provisional Level 1/2)")
select_level_1_or_2_tables()

## Clinically Grounded Candidate Features

Grounded in the literature review and the available C-OPN tables, the most promising feature groups for distinguishing `PD` from `AP` are:

- motor severity and asymmetry proxies from `mds-updrs.csv` and `timed_up_go.csv`
- cognition from `moca.csv`
- autonomic dysfunction from `scopa.csv`
- falls, disease course, and treatment response context from `clinical.csv` and `medication.csv`
- mood, apathy, fatigue, and quality-of-life burden from the questionnaire tables
- demographic and exposure variables from `demographic.csv` and `epidemiological.csv`

This notebook should be used to refine a lower-burden feature subset once the official implementation-level annotations become available.

In [ ]:
table_missingness_view = full_summary[[
    "filename",
    "row_count",
    "column_count",
    "cell_missing_pct",
    "column_missing_ge_50_pct",
    "likely_implementation_level",
]].sort_values(["cell_missing_pct", "column_count"], ascending=[False, False])

print("Most sparse tables")
display(table_missingness_view.head(10))

print("Least sparse tables")
display(table_missingness_view.sort_values(["cell_missing_pct", "column_count"]).head(10))

## Text-Aware Feature Selection

There are many non-numeric C-OPN columns, including categorical strings, bilingual answer labels, and free-text style responses such as `Other` fields. To handle that without downloading a large external model, the workflow below:

- concatenates non-numeric columns into a row-level text document using `column=value` tokens
- vectorizes that text with TF-IDF
- keeps numeric columns in parallel
- applies an L1-regularized linear selector to reduce the combined sparse feature space

This gives a lightweight feature-selection baseline that can absorb textual inputs while remaining easy to rerun.

In [ ]:
feature_selection_result = run_feature_selection_experiment(prepared_binary)

print("Feature selection summary")
display(pd.DataFrame([feature_selection_result.summary]))

print("Selected feature types")
display(feature_selection_result.selected_features["feature_type"].value_counts().rename_axis("feature_type").to_frame("n"))

print("Top selected features")
display(feature_selection_result.selected_features.head(25))

print("Selected-model metrics")
display(pd.DataFrame([{
    key: value for key, value in feature_selection_result.metrics.items() if key != "classification_report"
}]))

## EDA Takeaways

Key observations from the executed notebook should be:

- the dataset is wide, baseline-only, and highly sparse
- the cohort is strongly imbalanced toward `PD`, with `AP` as the rare positive class for the main task
- some raw headers collide after normalization unless the renaming logic enforces unique names per source file
- not all columns should be used; the combination of missingness filtering, domain-informed table selection, and sparse-model feature selection is more appropriate than a full-column brute-force approach
- text-derived signals from categorical/free-text responses can be incorporated without needing a heavyweight language model